# Lab 0: Bienvenida a la Qiskit Global Summer School 2026

Este es el Lab 0, tu notebook de calentamiento para prepararte para el evento principal. En este lab vas a:

- Actualizar tus paquetes de Python
- Completar ejercicios de programación sencillos con Sampler y Estimator
- Construir un circuito cuántico especial de varios qubits y explorar su vector de estado
- *(Opcional)* Construir un circuito cuántico en C usando la API de C de Qiskit

¿Emocionada o emocionado? ¡Hay mucho más por delante!


Este lab funciona en macOS, Linux y en plataformas en la nube como [qBraid](https://qbraid.com/) y [Google Colab](https://colab.research.google.com/). Si usas Windows, te recomendamos usar una plataforma en la nube para tener la mejor experiencia, sobre todo en los próximos labs, donde algunos paquetes funcionan mejor en entornos basados en Linux.

Un resumen rápido:

| Tu sistema operativo | Cómo ejecutar este lab |
|---|---|
| macOS / Linux | Ejecútalo localmente: el notebook se encarga de la configuración. |
| Windows | Los capítulos 1 y 2 funcionan localmente con Python y venv. Para el capítulo 3 opcional, abre este notebook en [qBraid](https://qbraid.com/) o [Google Colab](https://colab.research.google.com/). |

<br>

## Instalación

Ejecuta primero esta celda para revisar tu versión de Python.

In [ ]:
import sys

recommended = (3, 10)
current = sys.version_info[:2]

if current < recommended:
    print(f"Estás usando Python {current[0]}.{current[1]}. Se recomienda Python {recommended[0]}.{recommended[1]} o posterior.")
else:
    print(f"Python {current[0]}.{current[1]}: ¡todo bien!")

Después, ejecuta la celda de abajo para instalar y actualizar los paquetes necesarios. Si la actualización causa conflictos de dependencias con paquetes que ya tienes instalados en tu entorno local, __crea un entorno de Python nuevo__ (se recomienda Python 3.12) y continúa el desafío ahí.

Para obtener orientación sobre cómo preparar un entorno limpio e instalar Qiskit, consulta la documentación de IBM Quantum: https://quantum.cloud.ibm.com/docs/es/guides/install-qiskit

In [ ]:
# Instala los paquetes necesarios.
# Descomenta las líneas de abajo la primera vez que ejecutes este notebook para instalar las dependencias,
# y luego vuelve a comentarlas en las siguientes ejecuciones.


%pip install --upgrade 'qiskit[visualization]>=2.5.0' qiskit-aer qiskit-ibm-runtime matplotlib
%pip install --upgrade qc-grader

Cuando hayas instalado y actualizado las bibliotecas, reinicia el kernel de Python y vuelve a ejecutar el notebook empezando por la celda de importación de bibliotecas de abajo.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys

import qc_grader
import qiskit

from qc_grader.challenges.qgss_2026 import check_progress
from qc_grader.challenges.qgss_2026.lab0 import (
    grade_lab0_ex1,
    grade_lab0_ex2,
    grade_lab0_ex3,
    grade_lab0_ex4,
)


from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.visualization import array_to_latex, plot_histogram, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Batch, QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt

print(f"Versión de Qiskit: {qiskit.__version__}")
print(f"Versión del grader: {qc_grader.__version__}")

# Capítulo 1: tu primer programa cuántico con los patrones de Qiskit

## 1.1 Cuenta de IBM Quantum

¡Lo primero es lo primero! Para participar en el evento principal necesitarás una cuenta activa de IBM Quantum.

Si ya guardaste las credenciales de tu cuenta con `QiskitRuntimeService`, solo ejecuta la celda de abajo y aparecerá tu lista de *backends*. Si eres nuevo, o si hace tiempo que no lo configuras, sigue estos pasos para guardar tu cuenta:

1. Ve a [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com/) e inicia sesión (o crea una cuenta).
2. Copia tu clave de API haciendo clic en `Create +` o en `View all` en el panel principal.
3. Copia el CRN de tu instancia desde la pestaña Instances.
4. Reemplaza los marcadores de la celda de abajo con tu clave de API y el CRN de tu instancia, y ejecútala.

Cuando termines, ejecuta la celda del grader para confirmar que tus credenciales están bien configuradas.

In [ ]:
# Pega aquí tus credenciales de IBM Cloud. Solo necesitas hacerlo una vez por entorno.
# Cuando funcione, puedes comentar estas líneas para que las credenciales no queden en el archivo.

#token = "<YOUR_API_KEY>"
#instance = "<YOUR_CRN>"  # se ve como "crn:v1:bluemix:public:quantum-computing:..."

#QiskitRuntimeService.save_account(
#    token=token,
#    instance=instance,
#    overwrite=True,
#    set_as_default=True,
#)

# Verifica
service = QiskitRuntimeService()
backends = service.backends()
print(f"Cuenta OK. {len(backends)} backend(s) disponible(s):")
for b in backends[:5]:
    print(f"  {b.name} ({b.num_qubits} qubits)")

<div class="alert alert-block alert-success">
  <b>Ejercicio 1: verifica tu cuenta de IBM Quantum</b>

  ¿Ya configuraste tu clave de API y tu CRN, y revisaste los <i>backends</i> disponibles?
  Ejecuta la celda del Ejercicio 1 de abajo para comprobar que el grader funciona correctamente.
  ¡No necesitas escribir nada!

  Si tu cuenta está bien configurada, verás un mensaje de éxito.
</div>

In [ ]:
grade_lab0_ex1()

## 1.2 Computación cuántica con compuertas

Los computadores cuánticos de IBM se basan en compuertas. Construimos algoritmos cuánticos componiendo compuertas cuánticas, de forma parecida a como los computadores clásicos usan las compuertas AND, OR y NOT para procesar bits. Una diferencia clave es que las compuertas cuánticas actúan sobre qubits, la contraparte cuántica de los bits clásicos. Los bits clásicos son estrictamente 0 o 1, pero los qubits pueden mostrar tres fenómenos de la mecánica cuántica: la superposición (un qubit existe en una combinación lineal de 0 y 1 hasta que se mide), el entrelazamiento (dos o más qubits comparten un estado cuántico conjunto que no se puede describir con los qubits individuales de forma independiente) y la interferencia (las amplitudes cuánticas pueden sumarse de forma constructiva o destructiva, lo que permite a los algoritmos suprimir respuestas incorrectas y reforzar las correctas). Estos tres fenómenos son lo que hace que la computación cuántica sea distinta de la clásica.

### La superposición y la compuerta Hadamard

La [compuerta Hadamard (H)](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.library.HGate) toma un qubit en un estado definido ($|0\rangle$ o $|1\rangle$) y lo pone en una superposición equitativa de ambos:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad H|0\rangle = \frac{1}{\sqrt{2}}\bigl(|0\rangle + |1\rangle\bigr)$$

Cuando mides este estado, obtienes 0 o 1 con la misma probabilidad (50 % cada uno). Los coeficientes $\frac{1}{\sqrt{2}}$ son amplitudes de probabilidad, y la probabilidad de medir cada resultado es el cuadrado de su amplitud.

Completa la celda de abajo para construir tu primer circuito de 1 qubit con una compuerta Hadamard.

<br>

<div class="alert alert-block alert-info">

Para aprender más sobre la superposición, el entrelazamiento y la interferencia, consulta el curso [Basics of quantum information](https://quantum.cloud.ibm.com/learning/es/courses/basics-of-quantum-information).

</div>

<div class="alert alert-block alert-success">
<b>Ejercicio 2: construye tu primer circuito cuántico</b>

Abajo hay un circuito vacío de 1 qubit. Agrega una <b>compuerta Hadamard</b> en el qubit 0 para ponerlo en superposición. Esto es lo que produce la característica división 50/50 cuando ejecutamos el Sampler en la sección 1.6.

Pista: prueba con nuestro [Inicio rápido](https://quantum.cloud.ibm.com/docs/es/guides/quick-start) oficial.

</div>

In [ ]:
qc = QuantumCircuit(1)

# TODO: Agrega una compuerta Hadamard en el qubit 0

qc.draw("mpl")

## 1.3 ¿Qué es un patrón de Qiskit?

Ya tenemos un circuito cuántico que prepara el estado de superposición $H|0\rangle$. En las siguientes secciones haremos dos experimentos con este circuito: uno para obtener conteos de medición y otro para calcular el valor esperado del observable Z. Los dos experimentos siguen el mismo flujo de trabajo de cuatro pasos, llamado [patrón de Qiskit](https://quantum.cloud.ibm.com/docs/es/guides/intro-to-patterns).

| Paso | Qué haces | En este lab |
|------|------------|-------------|
| 1. Mapear | Mapear el problema a circuitos y operadores cuánticos. | Construimos el circuito con H y definimos el observable Z. |
| 2. Optimizar | Optimizar el circuito para el hardware de destino. | Usamos `generate_preset_pass_manager` para hacerlo. |
| 3. Ejecutar | Ejecutar en el hardware de destino usando primitivas. | Ejecutamos el circuito con Sampler y con Estimator. |
| 4. Post-procesar | Post-procesar los resultados. | Graficamos el histograma de mediciones y verificamos que ⟨Z⟩ ≈ 0. |

Para el paso de ejecución, Qiskit ofrece dos primitivas:

- [SamplerV2](https://quantum.cloud.ibm.com/docs/es/guides/primitives) muestrea el registro de salida de la ejecución de un circuito cuántico. La usamos para obtener conteos de medición y verificar que el 0 y el 1 aparecen con una frecuencia más o menos igual.
- [EstimatorV2](https://quantum.cloud.ibm.com/docs/es/guides/primitives) calcula valores esperados de observables respecto a los estados preparados por circuitos cuánticos. La usamos para evaluar $\langle Z \rangle$, donde $|0\rangle$ tiene autovalor $+1$ y $|1\rangle$ tiene autovalor $-1$, lo que da $\langle Z \rangle = 0$ para una superposición equitativa.

Las siguientes secciones recorren cada paso.

## 1.4 Paso 1: Mapear

En el paso de mapeo definimos el problema cuántico como un circuito y un observable.

El circuito ya está construido: `qc` prepara el estado $H|0\rangle$. Aquí definimos el observable. Usamos el operador Z (Pauli-Z), que tiene autovalor $+1$ para $|0\rangle$ y $-1$ para $|1\rangle$:

$$Z = |0\rangle\langle 0| - |1\rangle\langle 1|$$

Para el estado $H|0\rangle$, los dos resultados son igual de probables, así que el valor esperado es:

$$\langle Z \rangle = \frac{1}{2}(+1) + \frac{1}{2}(-1) = 0$$

Esto es lo que verificaremos con Estimator en el paso de ejecución.

<div class="alert alert-block alert-success">
<b>Ejercicio 3: define el observable</b>

En el paso de mapeo eliges qué medir. Queremos el operador Pauli-Z sobre el qubit 0; su valor esperado nos dice qué tan inclinado está el qubit hacia |0⟩ o hacia |1⟩.

Crea un __observable__ Z de un qubit usando <code>SparsePauliOp</code> con un coeficiente $1.0$.

Pista: encontrarás buenos ejemplos en nuestra [documentación de SparsePauliOp](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.quantum_info.SparsePauliOp).

</div>

In [ ]:
# TODO: Define el observable como el operador Z de un qubit
observable = None

print(f"Observable: {observable}")

## 1.5 Paso 2: Transpilar

El hardware cuántico tiene restricciones. El hardware cuántico real tiene un conjunto de compuertas nativas y una conectividad entre qubits determinados por su implementación física. El paso de transpilación (`Transpile`) reescribe tu circuito abstracto como un circuito equivalente que el *backend* de destino realmente puede ejecutar, usando las compuertas que soporta y su conectividad.

Usamos `generate_preset_pass_manager` para esto. En las celdas de abajo apuntamos a un simulador ideal, que acepta casi cualquier compuerta, así que el circuito transpilado se verá parecido al original. En la sección opcional 1.8 ejecutamos en hardware real, donde el transpilador trabaja más: descompone las compuertas en el conjunto de compuertas nativas del *backend* y optimiza la distribución de los qubits según las restricciones de conectividad.

La transpilación se aplica tanto al circuito como al observable. Cuando el transpilador asigna qubits virtuales a qubits físicos, el observable debe reflejar esa misma asignación. Esto lo hacemos con `apply_layout`. Como nuestro circuito tiene un solo qubit, aquí la distribución es trivial, pero el mismo patrón es obligatorio para circuitos de varios qubits en hardware real.

In [ ]:
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Sampler requiere mediciones en el circuito
qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm.run(qc_meas)

# Estimator recibe el circuito sin mediciones y evalúa el observable por separado
isa_qc = pm.run(qc)

# Aplica la misma distribución al observable para que los índices de los qubits coincidan con el circuito transpilado
isa_observable = observable.apply_layout(isa_qc.layout)

print("Circuito transpilado y observable listos.")

## 1.6 Paso 3: Ejecutar

Ahora ejecutamos el circuito transpilado con las dos primitivas. Cada una responde una pregunta distinta sobre el mismo estado cuántico.

### Sampler: "¿Qué conteos obtengo?"

El Sampler ejecuta tu circuito muchas veces (*shots*) y cuenta con qué frecuencia ocurre cada resultado: ¿cuántas veces mido 0 y cuántas veces mido 1?

In [ ]:
sampler = Sampler(mode=backend)
sampler_result = sampler.run([isa_qc_meas], shots=1000).result()
counts = sampler_result[0].data.meas.get_counts()

print("Conteos del Sampler:", counts)

<div class="alert alert-block alert-success">
<b>Envía y revisa tu resultado del Sampler para completar el Ejercicio 2</b>

Los <code>counts</code> que acabas de calcular vienen del circuito con H que construiste en la sección 1.2. Ejecuta la celda de abajo para enviarlos; el grader revisa que haya una división 50/50.

</div>

In [ ]:
grade_lab0_ex2(counts)

### Estimator: "¿Qué valor obtengo cuando observo mi circuito cuántico?"

Estimator recibe el circuito y el observable y devuelve directamente el valor esperado, sin que tengas que reunir y promediar conteos a mano. El circuito no necesita compuertas de medición: Estimator se encarga de medir internamente.

Para el estado $H|0\rangle$, esperamos $\langle Z \rangle = 0$.

In [ ]:
# Ejecuta Estimator: circuito SIN mediciones + observable
estimator = Estimator(mode=backend)
estimator_result = estimator.run([(isa_qc, isa_observable)]).result()
exp_val = estimator_result[0].data.evs

print(f"⟨Z⟩ = {exp_val:.4f}")

<div class="alert alert-block alert-success">
<b>Envía y revisa tu resultado del Estimator para completar el Ejercicio 3</b>

Ejecuta la celda de abajo para enviar <code>exp_val</code> al grader. Para una superposición H|0⟩ perfecta, ⟨Z⟩ debería ser ≈ 0.

</div>

In [ ]:
grade_lab0_ex3(exp_val)

## 1.7 Paso 4: Post-procesar

En el paso de post-procesamiento extraemos el significado de los resultados sin procesar. Según el problema, esto puede significar visualizar distribuciones de medición, calcular cantidades derivadas a partir de valores esperados, alimentar un ciclo de optimización clásica o aplicar técnicas de mitigación de errores.

En este lab lo mantenemos sencillo. Graficamos los conteos del Sampler como un histograma para ver con qué frecuencia se midió 0 y 1, y graficamos el resultado del Estimator como una barra con una barra de error que muestra la incertidumbre estadística de $\langle Z \rangle$. En el simulador esperamos conteos iguales y $\langle Z \rangle \approx 0$. En hardware real, las dos gráficas reflejarán el efecto del ruido de los *shots* y del dispositivo.

In [ ]:
# Resultados del Sampler
print(f"Conteos del Sampler: {counts}")
display(plot_histogram(counts, title="Distribución de mediciones de H|0⟩ (1000 shots)"))

# Resultados del Estimator
ev = estimator_result[0].data.evs.item()
std = estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel("Valor esperado")
ax.set_title("Observable Z sobre H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()

## 1.8 (Opcional) Ejecutar en hardware cuántico real

<div class="alert alert-block alert-info">
<b>Esta sección es opcional y no se califica.</b> Envía trabajos a un computador cuántico real y usa tu tiempo de QPU. Puedes saltarla y continuar con el capítulo 2.
</div>

Hasta ahora usamos un simulador. Ahora ejecutamos el mismo circuito en un procesador cuántico real de IBM. El código sigue el mismo flujo de los patrones de Qiskit. Lo único que cambia es el *backend*.

El hardware cuántico real funciona con restricciones físicas que los simuladores no tienen. Las compuertas se ejecutan como pulsos de microondas sobre qubits físicos, y cada operación tiene una fidelidad finita. Por eso, los conteos estarán cerca de 50/50 pero no serán exactos, y $\langle Z \rangle$ estará cerca de 0 con una incertidumbre medible. Comparar estos resultados con los del simulador te da una idea concreta de dónde se vuelve útil la mitigación de errores cuánticos.

Enviamos juntos los trabajos de Sampler y de Estimator usando [Batch](https://quantum.cloud.ibm.com/docs/es/guides/run-jobs-batch). Batch agrupa varios trabajos en una sola reserva del *backend* para que se ejecuten uno tras otro sin volver a hacer fila. Este trabajo usa aproximadamente 20 segundos de tiempo de QPU en dispositivos `Heron`. El tiempo de espera en la cola varía según la carga del *backend*. Si la celda se agota antes de imprimir los IDs de los trabajos, la celda siguiente muestra cómo recuperar los resultados con el ID del trabajo.

In [ ]:
# Por defecto, esta sección se ejecuta en hardware cuántico real.
# Para ahorrar tu tiempo de QPU para el evento principal, descomenta la línea del simulador con ruido de abajo
# y reemplaza real_backend por real_backend_sim en el resto de esta sección.

real_backend = service.least_busy()
# real_backend_sim = AerSimulator.from_backend(real_backend)  # simulador con ruido

print(f"Ejecutando en: {real_backend.name} ({real_backend.num_qubits} qubits)")

# Paso 2: Transpila para el backend de destino
pm_real = generate_preset_pass_manager(optimization_level=1, backend=real_backend)

qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm_real.run(qc_meas)

isa_qc_real = pm_real.run(qc)
isa_obs = observable.apply_layout(isa_qc_real.layout)

# Paso 3: Ejecuta; las dos primitivas corren en un solo Batch para no volver a hacer fila
with Batch(backend=real_backend):
    sampler = Sampler()
    sampler.options.environment.job_tags = ["qgss26"]
    sampler_job = sampler.run([isa_qc_meas], shots=1000)

    estimator = Estimator()
    estimator.options.environment.job_tags = ["qgss26"]
    estimator_job = estimator.run([(isa_qc_real, isa_obs)])

sampler_job_id = sampler_job.job_id()
estimator_job_id = estimator_job.job_id()

print(f"ID del trabajo del Sampler:   {sampler_job_id}")
print(f"ID del trabajo del Estimator: {estimator_job_id}")

<div class="alert alert-block alert-info">
<b>Nota de la traducción: alternativa sin cuenta</b>

La celda anterior necesita una cuenta de IBM Quantum. Si no la tienes, ejecuta la celda de abajo <b>en lugar de</b> la anterior. Es el mismo código, pero con <code>FakeTorino</code>: una copia local del procesador Heron <code>ibm_torino</code> (133 qubits), con su modelo de ruido. Las celdas siguientes funcionan igual.
</div>

In [ ]:
# Alternativa sin cuenta: ejecuta esta celda en lugar de la anterior.
from qiskit_ibm_runtime.fake_provider import FakeTorino

real_backend = FakeTorino()

print(f"Ejecutando en: {real_backend.name} ({real_backend.num_qubits} qubits)")

# Paso 2: Transpila para el backend de destino
pm_real = generate_preset_pass_manager(optimization_level=1, backend=real_backend)

qc_meas = qc.copy()
qc_meas.measure_all()
isa_qc_meas = pm_real.run(qc_meas)

isa_qc_real = pm_real.run(qc)
isa_obs = observable.apply_layout(isa_qc_real.layout)

# Paso 3: Ejecuta; las dos primitivas corren en un solo Batch para no volver a hacer fila
with Batch(backend=real_backend):
    sampler = Sampler()
    sampler.options.environment.job_tags = ["qgss26"]
    sampler_job = sampler.run([isa_qc_meas], shots=1000)

    estimator = Estimator()
    estimator.options.environment.job_tags = ["qgss26"]
    estimator_job = estimator.run([(isa_qc_real, isa_obs)])

sampler_job_id = sampler_job.job_id()
estimator_job_id = estimator_job.job_id()

print(f"ID del trabajo del Sampler:   {sampler_job_id}")
print(f"ID del trabajo del Estimator: {estimator_job_id}")

In [ ]:
# (Opcional) Si la celda anterior se agotó antes de imprimir los resultados, copia los
# IDs de los trabajos de Sampler y Estimator de su salida en las cadenas de abajo y ejecuta esta celda.
# Si no, déjala comentada.

# sampler_job_id = "<pega el id del trabajo del sampler>"
# estimator_job_id = "<pega el id del trabajo del estimator>"
# sampler_job = service.job(sampler_job_id)
# estimator_job = service.job(estimator_job_id)

In [ ]:
# Paso 4: Resultados
hw_counts = sampler_job.result()[0].data.meas.get_counts()
hw_estimator_result = estimator_job.result()
hw_exp_val = hw_estimator_result[0].data.evs

print(f"Conteos del Sampler en hardware: {hw_counts}")
print(f"⟨Z⟩ del Estimator en hardware:  {hw_exp_val:.4f}")

Compara los resultados lado a lado. El simulador da una división 50/50. Los resultados del hardware muestran el efecto del ruido: pequeñas desviaciones de ese valor.

In [ ]:
display(plot_histogram(
    [counts, hw_counts],
    legend=["Simulador", f"Hardware ({real_backend.name})"],
    title="H|0⟩: simulador frente a hardware real",
))


# Resultados del Estimator
ev = hw_estimator_result[0].data.evs.item()
std = hw_estimator_result[0].data.stds.item()
print(f"\nEstimator ⟨Z⟩ = {ev:.4f} ± {std:.4f}")

fig, ax = plt.subplots(figsize=(4, 5))
ax.bar(["⟨Z⟩"], [ev], yerr=[std], capsize=10, color="steelblue", width=0.4)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.2, 1.2)
ax.set_ylabel(f"Valor esperado en ({real_backend.name})")
ax.set_title("Observable Z sobre H|0⟩ (Estimator)")
plt.tight_layout()
plt.show()

# Capítulo 2: circuito de varios qubits y visualización

¡Buen trabajo en el capítulo 1! Construiste una superposición de un qubit, la ejecutaste con Sampler y con Estimator, y viste cómo se ve el mismo estado cuántico desde dos perspectivas de medición distintas, tanto en un simulador como (si lo intentaste) en hardware cuántico real.

Ahora construyamos algo más interesante. En este capítulo trabajarás con 6 qubits para construir paso a paso un circuito cuántico especial. Al final veremos el vector de estado que produce y lo visualizaremos con la [QSphere](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.visualization.plot_state_qsphere), una herramienta de visualización para varios qubits que muestra la probabilidad y la fase de cada estado de la base. Quizá reconozcas cómo se ve el estado final.

## 2.1 Construye un estado entrelazado de 6 qubits

En este ejercicio usarás tres compuertas:

- [H](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.library.HGate): pone un qubit en una superposición equitativa de $|0\rangle$ y $|1\rangle$ (esta ya la conoces).
- [X](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.library.XGate): invierte un qubit de $|0\rangle$ a $|1\rangle$ y viceversa. Es el equivalente cuántico de una compuerta NOT clásica.
- [CX](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.circuit.library.CXGate): una compuerta de dos qubits que invierte el qubit objetivo si y solo si el qubit de control está en $|1\rangle$. Aplicar CX a un estado de superposición crea entrelazamiento entre los dos qubits.

<div class="alert alert-block alert-info">

Si quieres explorar más estas compuertas y ejecutarlas en hardware real, consulta <a href="https://quantum.cloud.ibm.com/learning/es/courses/use-a-qc-today/build-and-run-your-first-quantum-program">Build and run your first quantum program</a>.

</div>


Todos los qubits empiezan en $|0\rangle$. Empieza con la celda de abajo.

<div class="alert alert-block alert-success">
<b>Ejercicio 4: construye un estado entrelazado de 6 qubits</b>

Construirás un circuito de 6 qubits. Cuando lo visualices más adelante en una QSphere, verás algo conocido.

Constrúyelo en tres pasos usando las celdas de abajo:

1. (ya está hecho) Crea un circuito de 6 qubits.
2. Aplica <b>H</b> al qubit 0 para ponerlo en superposición y luego aplica <b>X</b> al qubit 1.
3. Aplica una <b>cadena de CNOT</b> con el qubit 0 como control y los qubits 1 a 5 como objetivos para entrelazarlos a todos.

Pista: puedes usar un ciclo `for` para construir la cadena de CNOT.
</div>

In [ ]:
# Paso 1: Crea un circuito de 6 qubits
special_qc = QuantumCircuit(6)
special_qc.draw("mpl")

A continuación, aplica H al qubit 0 para ponerlo en superposición. Luego aplica X al qubit 1 para invertirlo de $|0\rangle$ a $|1\rangle$. Ahora el circuito tiene dos qubits preparados en estados específicos, listos para el entrelazamiento del siguiente paso.

In [ ]:
# Paso 2: H en el qubit 0, X en el qubit 1
# TODO: Aplica H al qubit 0

# TODO: Aplica X al qubit 1


special_qc.draw('mpl')

Ahora creamos el entrelazamiento con una cadena de CNOT. Cada compuerta CNOT usa el qubit 0 como control y tiene como objetivo los qubits 1 a 5. Esto une a todos los qubits: cuando el qubit 0 está en superposición, todos los demás qubits quedan entrelazados con él. El diagrama del circuito muestra cinco compuertas CNOT que conectan el qubit 0 con cada uno de los demás qubits.

In [ ]:
# Paso 3: Cadena de CNOT: entrelaza todos los qubits con el qubit 0
# TODO: Recorre los qubits 1 a 5 y aplica una CNOT con el qubit 0 como control


special_qc.draw('mpl')

## 2.2 Vector de estado

El vector de estado es la descripción matemática completa de un estado cuántico. Para $n$ qubits es un vector complejo de $2^n$ amplitudes, una por cada estado posible de la base. La probabilidad de medir un estado de la base en particular es el cuadrado de su amplitud.

Para nuestro circuito de 6 qubits, el vector de estado tiene $2^6 = 64$ entradas, que cubren todos los estados de la base desde $|000000\rangle$ hasta $|111111\rangle$. La clase `Statevector` lo calcula a partir del circuito, y `array_to_latex` lo imprime en forma matemática.

La mayoría de las entradas serán cero. Solo los estados de la base con amplitud distinta de cero contribuyen al estado cuántico que construiste.

In [ ]:
sv = Statevector(special_qc)
array_to_latex(sv, max_size=64)

## 2.3 Visualiza tu estado cuántico

Leer 64 números complejos directamente no es práctico. Necesitamos una forma de ver la estructura de un vistazo.

Para un solo qubit, la [esfera de Bloch](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.visualization.plot_bloch_multivector) representa cualquier estado puro como un punto sobre la superficie de una esfera. Pero solo funciona para un qubit a la vez, y seis esferas de Bloch separadas no pueden mostrar cómo están entrelazados los qubits.

La [QSphere](https://quantum.cloud.ibm.com/docs/es/api/qiskit/qiskit.visualization.plot_state_qsphere) pone todo el estado de varios qubits en una sola esfera. Cada estado de la base es un punto sobre la esfera, con tres propiedades:

- La latitud está determinada por el peso de Hamming (el número de 1 en la cadena de bits). Los estados sin ningún 1 están en el polo norte; los estados con todos sus bits en 1 están en el polo sur.
- El tamaño del punto representa la probabilidad de medir ese estado de la base. Los estados con amplitud cero no aparecen.
- El color representa la fase cuántica.

Para el estado que construiste verás dos puntos del mismo tamaño. Fíjate a qué estados de la base corresponden.

In [ ]:
# La QSphere
plot_state_qsphere(sv)

### Qué muestra la QSphere

La QSphere tiene dos puntos, uno cerca de la parte de arriba y otro cerca de la parte de abajo. Son los dos estados de la base en superposición equitativa, unidos por el entrelazamiento.

<div class="alert alert-block alert-success">
<b>Ejercicio 4: envía tu vector de estado</b>

Envía tu <code>sv</code> al grader para completar el Ejercicio 4. El grader revisará tu circuito y te revelará por qué este estado es especial. También recibirás enlaces a cursos que te ayudarán a prepararte para el evento principal. Ejecuta la celda de abajo para terminar el Lab 0.

</div>

In [ ]:
grade_lab0_ex4(sv)

### Qué sigue

IBM puso su primer computador cuántico en la nube el 4 de mayo de 2016. Desde entonces, Qiskit y la comunidad de computación cuántica han crecido juntos con cada generación de hardware, software y educación. Qiskit lo construye su comunidad, y te acompañará durante todo tu camino cuántico.

En este lab ejecutaste un circuito cuántico en un simulador y en hardware real, mediste un estado cuántico con dos primitivas distintas y visualizaste un estado entrelazado de 6 qubits. Estos son los bloques básicos que usarás a lo largo de la QGSS 2026.

Cerremos los ejercicios principales revisando tu progreso. El capítulo 3 tiene un ejercicio opcional en el que puedes probar otra forma de usar Qiskit: construir circuitos cuánticos en C. Pásate por ahí cuando quieras.

In [ ]:
check_progress("lab0")

# Capítulo 3 (opcional, no se califica): construye un circuito cuántico en C

<div class="alert alert-block alert-info">
<b>Este capítulo es opcional y no se califica.</b> Requiere un compilador de C (GCC o Clang). Si usas Windows, consulta la sección 3.1 antes de empezar.
</div>

En los capítulos anteriores construiste circuitos cuánticos en Python. Qiskit también ofrece una [API de C](https://www.ibm.com/quantum/blog/c-api-enables-end-to-end-hpc-demo) que permite construir circuitos en C y pasarlos de vuelta a Python. Esto es útil cuando los programas cuánticos se ejecutan junto con cargas de trabajo clásicas de computación de alto rendimiento (HPC) escritas en lenguajes compilados.

Este capítulo sigue la guía [Extiende Qiskit en Python con C](https://quantum.cloud.ibm.com/docs/es/guides/c-extension-for-python). Reconstruirás en C el circuito del capítulo 2 y luego usarás Python para calcular el vector de estado.

<div class="alert alert-block alert-warning">
La API de C de Qiskit es experimental y requiere que la versión con la que compilas coincida con la versión menor del <i>runtime</i> instalado. Como este notebook instala la versión más reciente de Qiskit, la compilación y el <i>runtime</i> siempre están sincronizados. Consulta la <a href="https://quantum.cloud.ibm.com/docs/es/guides/c-extension-for-python">guía de extensiones en C</a> de IBM para más detalles.
</div>

## 3.1 Configuración

Los encabezados y la biblioteca de la API de C de Qiskit vienen incluidos cuando instalas Qiskit con pip, así que no hace falta una descarga aparte. Para compilar una extensión en C necesitas un compilador de C y `setuptools`.

En macOS, Linux, Google Colab y qBraid todo funciona sin configuración adicional. Ejecuta la celda de configuración de abajo para empezar.

<details>
<summary><b>Usuarios de Windows</b></summary>

<br>

El capítulo 3 no tiene soporte oficial en Windows. Los entornos de Windows varían mucho en la cadena de herramientas del compilador, la versión de MSVC, la disponibilidad de BLAS y el manejo de rutas, así que un solo notebook no puede garantizar una compilación sin problemas en todas las configuraciones.

Para una experiencia sin fricciones, abre este notebook en <a href="https://qbraid.com/">qBraid</a> o en <a href="https://colab.research.google.com/">Google Colab</a>, donde ya hay un compilador de C disponible.

</details>

<div class="alert alert-block alert-info">
Si ya tienes GCC o Clang instalado y prefieres configurarlo a mano, puedes saltarte la celda de configuración y seguir directamente la documentación de IBM.

<ul>
<li><a href="https://quantum.cloud.ibm.com/docs/es/guides/install-c-api">Instalar la API de C de Qiskit</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/es/guides/c-extension-for-python">Extiende Qiskit en Python con C</a></li>
<li><a href="https://quantum.cloud.ibm.com/docs/es/api/qiskit-c">Referencia de la API de C de Qiskit</a></li>
</ul>
</div>

In [ ]:
import sys
import os
import platform
import shutil
import subprocess

from qiskit.quantum_info import Statevector
from qiskit.visualization import array_to_latex, plot_state_qsphere

# Configuración para macOS / Linux / Colab / qBraid.
# En Windows, consulta la nota de la sección 3.1 y usa qBraid o Google Colab.

if platform.system() == "Windows":
    print("Windows: ejecuta este capítulo en qBraid o en Google Colab.")
else:
    IS_COLAB = "google.colab" in sys.modules

    # Colab no trae un compilador por defecto.
    if IS_COLAB and shutil.which("gcc") is None:
        subprocess.run(["apt-get", "-qq", "install", "-y", "build-essential"], check=True)

    # setuptools compila el archivo de C como un módulo de Python.
    try:
        import setuptools  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools"], check=True)

    cc = shutil.which("gcc") or shutil.which("clang")
    if cc is None:
        print("No se encontró un compilador de C. En macOS ejecuta: xcode-select --install")
    else:
        print("Compilador de C:", cc)
        print("Configuración lista. Continúa con la sección 3.2.")

## 3.2 Construye en C el circuito del logo

Las funciones de la API de C que usarás en este ejercicio:

| Función | Qué hace |
|---|---|
| `qk_circuit_new(num_qubits, num_clbits)` | Crea un circuito nuevo |
| `qk_circuit_gate(qc, QkGate_H, qubits, NULL)` | Agrega una compuerta. `qubits` es un arreglo `uint32_t` con los índices de los qubits. Usa `QkGate_H`, `QkGate_X` o `QkGate_CX`. |
| `qk_circuit_measure(qc, qubit, clbit)` | Agrega una medición |
| `qk_circuit_to_python_full(qc)` | Convierte el circuito de C en un `QuantumCircuit` de Python y transfiere su propiedad. No llames a `qk_circuit_free` después de esto. |

Dos reglas que aplican a toda extensión en C que use la API de C de Qiskit:

- Define `QISKIT_PYTHON_EXTENSION` e incluye `Python.h` antes de `qiskit.h`.
- Llama a `qk_import()` una vez en `PyInit_<nombre>`. Sin esto, las llamadas a la API fallan en tiempo de ejecución.

Para ver un ejemplo completo con todo el código base, consulta la [guía de extensiones en C](https://quantum.cloud.ibm.com/docs/es/guides/c-extension-for-python).

<div class="alert alert-block alert-info">
Python carga un módulo compilado una sola vez por sesión. Si editas el archivo de C y vuelves a compilar, reinicia el kernel antes de importarlo otra vez.
</div>

<div class="alert alert-block alert-success">
<b>Ejercicio opcional: construye en C el circuito del logo</b>

Construye el mismo circuito de 6 qubits del Ejercicio 4, esta vez en C. Completa los tres pasos de compuertas en el archivo de abajo.

1. Aplica <b>H</b> al qubit 0.
2. Aplica <b>X</b> al qubit 1.
3. Agrega una <b>cadena de CNOT</b>: el control es el qubit 0 y los objetivos son los qubits 1 a 5. Aquí sirve un ciclo <code>for</code>.

El circuito no tiene mediciones, así que su vector de estado se puede calcular en el paso siguiente.
</div>

In [ ]:
%%writefile qgss_circuit.c

#include <Python.h>
#include <qiskit.h>
#include <stdint.h>

static PyObject *build_special_circuit(PyObject *self, PyObject *args) {
    QkCircuit *qc = qk_circuit_new(6, 0);

    // TODO 1: Aplica H al qubit 0.
    //   Usa qk_circuit_gate con QkGate_H. Consulta la referencia de la API de C.

    // TODO 2: Aplica X al qubit 1.
    //   Usa qk_circuit_gate con QkGate_X.

    // TODO 3: Agrega una cadena de CNOT. El control es el qubit 0 y los objetivos son 1 a 5.
    //   Para cada objetivo, llama a qk_circuit_gate con QkGate_CX y un arreglo {control, objetivo}.
    //   Aquí sirve un ciclo for.

    return qk_circuit_to_python_full(qc);
}

static PyMethodDef methods[] = {
    {"build_special_circuit", build_special_circuit, METH_NOARGS, "Build the special circuit."},
    {NULL, NULL, 0, NULL},
};
static struct PyModuleDef moduledef = {
    .m_base = PyModuleDef_HEAD_INIT,
    .m_name = "qgss_circuit",
    .m_methods = methods,
};
PyMODINIT_FUNC PyInit_qgss_circuit(void) {
    if (qk_import() < 0) {
        return NULL;
    }
    return PyModuleDef_Init(&moduledef);
}

Cuando hayas completado los TODO, ejecuta la celda de abajo para compilar la extensión.

In [ ]:
%%writefile setup.py
from setuptools import setup, Extension
import qiskit.capi
import sys
import subprocess

def supports_no_warn_duplicate():
    try:
        result = subprocess.run(
            ["ld", "-no_warn_duplicate_libraries"],
            capture_output=True
        )
        return b"unknown option" not in result.stderr
    except Exception:
        return False

extra_link_args = []
if sys.platform == "darwin" and supports_no_warn_duplicate():
    extra_link_args = ["-Wl,-no_warn_duplicate_libraries"]

setup(
    name="qgss_circuit",
    ext_modules=[
        Extension(
            name="qgss_circuit",
            sources=["qgss_circuit.c"],
            include_dirs=[qiskit.capi.get_include()],
            define_macros=[("QISKIT_PYTHON_EXTENSION", "1")],
            extra_link_args=extra_link_args,
        )
    ],
)

`setup.py build_ext --inplace` compila el archivo de C y deja el módulo resultante en el directorio actual, listo para importarlo.

In [ ]:
# Compila el archivo de C como un módulo de Python.
subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], check=True)
print("Compilación terminada.")
print("Si ya habías ejecutado esta celda antes, reinicia el kernel (Kernel > Restart) antes de importar.")

## 3.3 Lee el estado en Python

La API de C de Qiskit construye circuitos, pero no los simula. Para obtener el vector de estado, pasa el circuito de vuelta a Python y usa las mismas herramientas del capítulo 2.

In [ ]:
import qgss_circuit

special_qc_c = qgss_circuit.build_special_circuit()
sv = Statevector(special_qc_c)
array_to_latex(sv, max_size=64)

In [ ]:
plot_state_qsphere(sv, show_state_labels=True)

Como construiste el mismo circuito del capítulo 2, tu vector de estado final debería aprobar el Ejercicio 4 en el grader si el circuito quedó bien armado.
Si no lo apruebas, ¡no te preocupes! Muchas personas ya aprobaron el grader antes, así que, aunque te equivoques esta vez, el resultado final de tu revisión de progreso no cambiará. Recuerda que este es un ejercicio __opcional__.

In [ ]:
grade_lab0_ex4(sv)

In [ ]:
check_progress("lab0")

# Información adicional

**Creado por:** Sophy Shin

**Versión:** 1.0.0

*Traducción al español para el Qiskit Fall Fest 2026.*